# CropMedic Batch 1 — Common Bean pilot

This notebook trains the **audited BLD-BD Common Bean pilot dataset**. It is designed to survive Colab disconnects: the dataset ZIP, checkpoints and final artifacts are persisted in Google Drive.

Expected classes: **Angular Leaf Spot, Bean Common Mosaic Virus, Healthy** — 224 original images per class, 672 total.

This remains a pilot model. Do **not** mark Common bean as AI-supported until independent field validation and rejection testing pass.

In [ ]:
!nvidia-smi
%cd /content
!test -d CropMedic/.git && git -C CropMedic pull origin production-v1 || git clone -b production-v1 https://github.com/macdonaldchigutiro/CropMedic.git
%cd /content/CropMedic/crop_medic_backend/training
!pip -q install -r requirements-training.txt

## Mount Google Drive

Drive is used before training so a Colab disconnect does not wipe the dataset ZIP, checkpoints or finished model.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/CropMedic')
dataset_dir = root / 'datasets'
checkpoint_dir = root / 'checkpoints/common_bean'
artifact_dir = root / 'batch1/common_bean'
dataset_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print('Persistent root:', root)

## Upload once, then reuse from Drive

The first time, choose `common_bean_for_colab.zip`. After that, rerunning the notebook reuses the Drive copy automatically.

In [ ]:
from google.colab import files
import shutil, zipfile

drive_zip = dataset_dir / 'common_bean_for_colab.zip'
if not drive_zip.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one ZIP: common_bean_for_colab.zip')
    zip_name = next(iter(uploaded))
    if not zip_name.lower().endswith('.zip'):
        raise RuntimeError('The uploaded file must be a ZIP archive.')
    shutil.copy2(zip_name, drive_zip)
    print('Saved dataset ZIP to Drive:', drive_zip)
else:
    print('Reusing dataset ZIP from Drive:', drive_zip)

target = Path('data_batch1/common_bean')
if target.exists():
    shutil.rmtree(target)
target.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(drive_zip) as z:
    z.extractall(target)
print('Extracted to', target.resolve())

In [ ]:
!python audit_crop_dataset.py --data-dir data_batch1/common_bean --crop-id common_bean

classes = sorted([p.name for p in Path('data_batch1/common_bean').iterdir() if p.is_dir()])
image_count = sum(1 for p in Path('data_batch1/common_bean').rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
print('Classes:', classes)
print('Images:', image_count)
expected = ['Angular Leaf Spot', 'Bean Common Mosaic Virus', 'Healthy']
if classes != expected or image_count != 672:
    raise RuntimeError('Dataset does not match the audited 672-image Common Bean pilot. Stop before training.')

## Resumable training

If Colab disconnects during training, reconnect and rerun the notebook from the top. The trainer will reuse completed phases and restore an interrupted phase from the Drive checkpoint instead of starting from zero.

In [ ]:
!python train_crop_model.py --crop-id common_bean --data-dir data_batch1/common_bean --output-dir /content/drive/MyDrive/CropMedic/batch1/common_bean --checkpoint-dir /content/drive/MyDrive/CropMedic/checkpoints/common_bean --head-epochs 10 --fine-epochs 18

In [ ]:
import json
metrics_path = Path('/content/drive/MyDrive/CropMedic/batch1/common_bean/metrics.json')
if not metrics_path.exists():
    raise RuntimeError('Training has not finished yet; metrics.json is not present.')
m = json.load(open(metrics_path))
print('TEST ACCURACY:', m['test_accuracy'])
print('MACRO F1:', m['macro_f1'])
print(json.dumps(m['per_class'], indent=2))

In [ ]:
print('Artifacts:', artifact_dir)
print('Checkpoints:', checkpoint_dir)
print('Dataset ZIP:', drive_zip)
print('All three are persistent in Google Drive.')

## What happens after training

Freeze these pilot metrics before using a separate field dataset for external validation. Do not tune on the external validation set. Common bean remains **Coming soon** in the app until field validation and uncertainty/rejection checks pass.